# 02 — Generator Sanity Check
Runs CTGAN, TVAE, and CopulaGAN on a 5,000-row slice of each dataset.
Goal: confirm synthetic output is plausible before full training runs.
Checks: valid categories, sensible numeric ranges, no NaN leakage.

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable from notebooks/

import yaml
import pandas as pd
import numpy as np

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

SLICE_SIZE = 5000
SANITY_EPOCHS = 5   # small epoch count for fast iteration
SAMPLE_SIZE  = 500

print('Config loaded. Slice size:', SLICE_SIZE, '| Epochs:', SANITY_EPOCHS)

In [ ]:
from src.generators import GENERATOR_MAP

def run_sanity_check(dataset_name, generator_name, config, slice_size, epochs, sample_size):
    print(f'\n{"-"*60}')
    print(f'Dataset: {dataset_name} | Generator: {generator_name}')
    print(f'{"-"*60}')

    # Temporarily override epochs for speed
    sanity_config = yaml.safe_load(yaml.dump(config))  # deep copy
    sanity_config['generators'][generator_name]['epochs'] = epochs

    # Instantiate generator and load data
    gen_cls = GENERATOR_MAP[generator_name]
    gen = gen_cls(sanity_config, dataset_name)
    train_df = gen.load_train_data()

    # Slice
    slice_df = train_df.sample(n=min(slice_size, len(train_df)), random_state=42)
    print(f'Slice shape: {slice_df.shape}')

    # Fit
    metadata = gen.build_metadata(slice_df)
    gen.fit(slice_df, metadata)

    # Sample
    synthetic = gen.sample(sample_size)
    print(f'Synthetic shape: {synthetic.shape}')

    # --- Checks ---

    # 1. NaN check
    null_counts = synthetic.isnull().sum()
    null_cols = null_counts[null_counts > 0]
    print(f'\n[CHECK] NaN columns: {"None" if null_cols.empty else null_cols.to_dict()}')

    # 2. Categorical validity check
    cat_cols = slice_df.select_dtypes(include='category').columns
    invalid_cats = {}
    for col in cat_cols:
        valid_vals = set(slice_df[col].dropna().unique())
        synth_vals = set(synthetic[col].dropna().unique())
        out_of_vocab = synth_vals - valid_vals
        if out_of_vocab:
            invalid_cats[col] = out_of_vocab
    print(f'[CHECK] Invalid categories: {"None" if not invalid_cats else invalid_cats}')

    # 3. Numeric range check
    num_cols = slice_df.select_dtypes(include='number').columns
    range_violations = {}
    for col in num_cols:
        real_min, real_max = slice_df[col].min(), slice_df[col].max()
        synth_min, synth_max = synthetic[col].min(), synthetic[col].max()
        margin = (real_max - real_min) * 0.2  # allow 20% margin outside real range
        if synth_min < real_min - margin or synth_max > real_max + margin:
            range_violations[col] = {
                'real': (round(real_min, 2), round(real_max, 2)),
                'synth': (round(synth_min, 2), round(synth_max, 2))
            }
    print(f'[CHECK] Range violations: {"None" if not range_violations else range_violations}')

    # 4. Target distribution comparison
    target_col = config['datasets'][dataset_name]['target_col']
    print(f'\n[CHECK] Target distribution')
    print(f'  Real:      {slice_df[target_col].value_counts(normalize=True).round(3).to_dict()}')
    print(f'  Synthetic: {synthetic[target_col].value_counts(normalize=True).round(3).to_dict()}')

    print(f'\n[PASSED] {dataset_name} × {generator_name} sanity check complete')
    return synthetic

## Diabetes 130-US

In [ ]:
for gen_name in ['ctgan', 'tvae', 'copulagan']:
    run_sanity_check('diabetes_130us', gen_name, config, SLICE_SIZE, SANITY_EPOCHS, SAMPLE_SIZE)

## Home Credit

In [ ]:
for gen_name in ['ctgan', 'tvae', 'copulagan']:
    run_sanity_check('home_credit', gen_name, config, SLICE_SIZE, SANITY_EPOCHS, SAMPLE_SIZE)

## ACS Income

In [ ]:
for gen_name in ['ctgan', 'tvae', 'copulagan']:
    run_sanity_check('acs_income', gen_name, config, SLICE_SIZE, SANITY_EPOCHS, SAMPLE_SIZE)